In [1]:

# Import the pandas library
import pandas as pd

# Import the operating system library
import os

# Display pandas version
print("Pandas version:", pd.__version__)


Pandas version: 2.2.3


In [2]:

# Dataset URL
url = "https://github.com/dbdmg/data-science-lab/raw/master/datasets/online_retail.csv"

# Read the CSV file
df = pd.read_csv(url, encoding="ISO-8859-1")

# Display a success message
print("Dataset downloaded successfully!")


Dataset downloaded successfully!


In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [4]:

# Number of rows and columns
print("Dataset shape:", df.shape)

# Display column names
print("\nColumn names:")
print(df.columns.tolist())


Dataset shape: (541909, 8)

Column names:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [5]:

# Display information about the dataset
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [6]:

# Count missing values in each column
df.isnull().sum()


,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


In [7]:

# Count duplicate rows
print("Duplicate rows:", df.duplicated().sum())


Duplicate rows: 5268


In [8]:

# Summary of numerical columns
df.describe()


,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [9]:

# Create a copy of the original dataset
df_clean = df.copy()

# Remove duplicate rows
df_clean = df_clean.drop_duplicates()

# Check remaining duplicates
print("Duplicates after cleaning:",
      df_clean.duplicated().sum())


Duplicates after cleaning: 0


In [10]:

# Remove rows with missing product descriptions
df_clean = df_clean.dropna(subset=["Description"])

print("Missing descriptions:",
      df_clean["Description"].isnull().sum())


Missing descriptions: 0


In [11]:

# Check remaining missing CustomerID values
print("Missing CustomerID:",
      df_clean["CustomerID"].isnull().sum())


Missing CustomerID: 133583


In [12]:

# Convert InvoiceDate to datetime format
df_clean["InvoiceDate"] = pd.to_datetime(
    df_clean["InvoiceDate"],
    format="%m/%d/%Y %H:%M",
    errors="coerce"
)

# Check data type
print(df_clean["InvoiceDate"].dtype)


datetime64[ns]


In [13]:

# Calculate revenue for each transaction
df_clean["Revenue"] = (
    df_clean["Quantity"] * df_clean["UnitPrice"]
)

# Display the first 5 rows
df_clean[["Description", "Quantity",
          "UnitPrice", "Revenue"]].head()


,Description,Quantity,UnitPrice,Revenue
0,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,15.30
1,WHITE METAL LANTERN,6,3.39,20.34
2,CREAM CUPID HEARTS COAT HANGER,8,2.75,22.00
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,20.34
4,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,20.34


In [14]:

# Calculate total revenue
total_revenue = df_clean["Revenue"].sum()

print("Total Revenue: £", round(total_revenue, 2))


Total Revenue: £ 9726006.95


In [15]:

total_quantity = df_clean["Quantity"].sum()

print("Total Quantity Sold:", total_quantity)


Total Quantity Sold: 5176111


In [16]:

top_products = (
    df_clean.groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_products)


Description
DOTCOM POSTAGE                        206245.48
REGENCY CAKESTAND 3 TIER              164459.49
WHITE HANGING HEART T-LIGHT HOLDER     99612.42
PARTY BUNTING                          98243.88
JUMBO BAG RED RETROSPOT                92175.79
RABBIT NIGHT LIGHT                     66661.63
POSTAGE                                66230.64
PAPER CHAIN KIT 50'S CHRISTMAS         63715.24
ASSORTED COLOUR BIRD ORNAMENT          58792.42
CHILLI LIGHTS                          53746.66
Name: Revenue, dtype: float64


In [17]:

# Identify cancelled invoices
cancelled = df_clean[
    df_clean["InvoiceNo"].astype(str).str.startswith("C")
]

print("Cancelled transactions:", len(cancelled))


Cancelled transactions: 9251


In [18]:

# Keep valid sales transactions
df_sales = df_clean[
    (~df_clean["InvoiceNo"].astype(str).str.startswith("C")) &
    (df_clean["Quantity"] > 0) &
    (df_clean["UnitPrice"] > 0)
].copy()

# Calculate sales revenue
df_sales["Revenue"] = (
    df_sales["Quantity"] * df_sales["UnitPrice"]
)

print("Sales dataset shape:", df_sales.shape)
print("Sales revenue: £", round(df_sales["Revenue"].sum(), 2))


Sales dataset shape: (524878, 9)
Sales revenue: £ 10642110.8


In [19]:

# Check missing values
print(df_sales.isnull().sum())

# Check duplicates
print("Duplicates:", df_sales.duplicated().sum())

# Display first five records
df_sales.head()


InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     132186
Country             0
Revenue             0
dtype: int64
Duplicates: 0


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [20]:

# Create a folder for processed data
os.makedirs("data", exist_ok=True)

# Save the cleaned sales dataset
df_sales.to_csv(
    "data/cleaned_ecommerce_sales.csv",
    index=False
)

print("CSV file saved successfully!")


CSV file saved successfully!


In [21]:

from google.colab import files

files.download("data/cleaned_ecommerce_sales.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>